### Building a GPT

In [22]:
import tiktoken
import torch

In [23]:
import os

path = 'C:\ptoh\DeepLLM\data\input.txt'
if os.path.exists(path):
    with open(path, 'r', encoding='utf-8') as f:
        text = f.read()
else:
    text = ''
    print(f"No file found at {path}. Using empty fallback text.")

<>:3: SyntaxWarning: invalid escape sequence '\p'
<>:3: SyntaxWarning: invalid escape sequence '\p'
C:\Users\Abhishek Raj\AppData\Local\Temp\ipykernel_28540\3289695073.py:3: SyntaxWarning: invalid escape sequence '\p'
  path = 'C:\ptoh\DeepLLM\data\input.txt'


In [24]:
print(f"length of dataset in characters : {len(text)}")
print(text[:100])

length of dataset in characters : 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [25]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("".join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [26]:
# creating mappings from chars to integers
stoi  = {ch:i for i , ch in enumerate(chars)}
itos = {i:ch for i , ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]   #takes a string outputs a list of integers 
decode = lambda l: "".join(itos[i] for i in l) #takes a list of integers and outputs a string

print(encode("hi there"))
print(decode([46, 47, 1, 58, 46, 43, 56, 43]))

[46, 47, 1, 58, 46, 43, 56, 43]
hi there


In [27]:
data = torch.tensor(encode(text) , dtype = torch.long)
print(data.shape,data.dtype)
print(data[:10])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47])


In [28]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [29]:
# the maximum length of chucks fed into the transformer is the block_size or context length
block_size = 8
train_data[:block_size+1]


tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

in the context of 18 47 comes next in the context of 18,47 56 comes next

In [30]:
x = train_data[:block_size]
y= train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input : {context} the target : {target}")

when input : tensor([18]) the target : 47
when input : tensor([18, 47]) the target : 56
when input : tensor([18, 47, 56]) the target : 57
when input : tensor([18, 47, 56, 57]) the target : 58
when input : tensor([18, 47, 56, 57, 58]) the target : 1
when input : tensor([18, 47, 56, 57, 58,  1]) the target : 15
when input : tensor([18, 47, 56, 57, 58,  1, 15]) the target : 47
when input : tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target : 58


Now we introduce a batch dimension

In [31]:
torch.manual_seed(1337)
batch_size = 4 
block_size = 8

def get_batch(split):
    # generate a small batches of data of input x and target y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data)-block_size , (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i + 1 : i + block_size +1] for i in ix])
    return x , y 

xb , yb = get_batch('train')
print('inputs:')
print(xb.shape)
print('targets:')
print(yb.shape)
print(yb)
print('--'*5)

for b in range(batch_size):
    for i in range(block_size):
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target :{target}")

inputs:
torch.Size([4, 8])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----------
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target :39
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target :39
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target :39
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target :39
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target :39
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target :39
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target :39
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target :39
when input is [44, 53, 56, 1, 58, 46, 39, 58] the target :1
when input is [44, 53, 56, 1, 58, 46, 39, 58] the target :1
when input is [44, 53, 56, 1, 58, 46, 39, 58] the target :1
when input is [44, 53, 56, 1, 58, 46, 39, 58] the target :1
when input is [44, 53, 56, 1, 58, 46, 39, 58]